# 🧠 CNN v2 – Wełyczko
**Projekt: Vision Transformer vs CNN | GRUPA 3**

## Zadanie na następną lekcję
Rozbudowanie modelu CNN o opcjonalne komponenty i stworzenie funkcji parametrycznej:
- [x] Podstawowy CNN – już działa ~76% accuracy (Projekt1.ipynb)
- [ ] Dodać opcjonalny **BatchNorm** po warstwach konwolucyjnych
- [ ] Dodać opcjonalny **Dropout** w warstwie gęstej
- [ ] Dodać konfigurowalny **rozmiar poolingu** (MaxPool)
- [ ] Dodać konfigurowalną **liczbę detektorów** (filtrów)
- [ ] Stworzyć funkcję `run_cnn(epochs, batchnorm, dropout, pooling_size, n_filters, data)` → wyniki

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import numpy as np

## 1. Model CNN z opcjonalnymi komponentami

In [ ]:
class CifarCNN_v2(nn.Module):
    def __init__(self, num_classes=10, n_filters=64, use_batchnorm=False,
                 use_dropout=False, dropout_rate=0.5, pool_size=2):
        """
        Args:
            num_classes  : liczba klas (10 dla CIFAR-10)
            n_filters    : liczba filtrów w pierwszej warstwie konwolucyjnej
            use_batchnorm: czy dodać BatchNorm po Conv
            use_dropout  : czy dodać Dropout w warstwie gęstej
            dropout_rate : współczynnik dropout (jeśli use_dropout=True)
            pool_size    : rozmiar okna MaxPool
        """
        super().__init__()

        # Blok konwolucyjny 1
        layers1 = [nn.Conv2d(3, n_filters, kernel_size=3, padding=1)]
        if use_batchnorm:
            layers1.append(nn.BatchNorm2d(n_filters))  # TODO: sprawdź czy n_filters to poprawna liczba kanałów
        layers1 += [nn.ReLU(), nn.MaxPool2d(pool_size)]
        self.block1 = nn.Sequential(*layers1)

        # Blok konwolucyjny 2
        layers2 = [nn.Conv2d(n_filters, n_filters * 2, kernel_size=3, padding=1)]
        if use_batchnorm:
            layers2.append(nn.BatchNorm2d(n_filters * 2))
        layers2 += [nn.ReLU(), nn.MaxPool2d(pool_size)]
        self.block2 = nn.Sequential(*layers2)

        # TODO: obliczyć poprawny rozmiar flatten w zależności od pool_size i img_size (32x32)
        # Przy pool_size=2: 32 -> 16 -> 8, flatten = 8 * 8 * (n_filters*2)
        # Przy pool_size=4: 32 -> 8 -> 2,  flatten = 2 * 2 * (n_filters*2)
        flat_size = (32 // (pool_size ** 2)) ** 2 * (n_filters * 2)

        # Warstwa gęsta
        dense_layers = [nn.Flatten(), nn.Linear(flat_size, 512), nn.ReLU()]
        if use_dropout:
            dense_layers.append(nn.Dropout(dropout_rate))
        dense_layers.append(nn.Linear(512, num_classes))
        self.classifier = nn.Sequential(*dense_layers)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)

## 2. Funkcja parametryczna run_cnn()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def run_cnn(epochs, use_batchnorm, use_dropout, pool_size, n_filters,
            train_loader, test_loader, num_classes=10, lr=0.001):
    """
    Trenuje CNN z podanymi hiperparametrami i zwraca wyniki.

    Returns:
        dict z kluczami:
          'test_acc'      : końcowa dokładność na zbiorze testowym (%)
          'train_acc'     : końcowa dokładność na zbiorze treningowym (%)
          'train_time_s'  : czas treningu w sekundach
          'num_params'    : liczba parametrów modelu
          'history'       : lista słowników {epoch, train_acc, test_acc, loss} per epoka
    """
    model = CifarCNN_v2(
        num_classes=num_classes,
        n_filters=n_filters,
        use_batchnorm=use_batchnorm,
        use_dropout=use_dropout,
        pool_size=pool_size
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Liczba parametrów: {num_params:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        # --- Trening ---
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

        train_acc = 100 * correct_train / total_train

        # --- Ewaluacja ---
        model.eval()
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                correct_test += (predicted == labels).sum().item()
                total_test += labels.size(0)

        test_acc = 100 * correct_test / total_test
        avg_loss = running_loss / len(train_loader)

        print(f"Epoka [{epoch}/{epochs}] | Loss: {avg_loss:.4f} | "
              f"Train: {train_acc:.2f}% | Test: {test_acc:.2f}%")

        history.append({'epoch': epoch, 'train_acc': train_acc,
                        'test_acc': test_acc, 'loss': avg_loss})

    train_time = time.time() - start_time
    print(f"\nTrening zakończony w czasie: {train_time:.2f}s")

    return {
        'test_acc':     test_acc,
        'train_acc':    train_acc,
        'train_time_s': train_time,
        'num_params':   num_params,
        'history':      history,
    }

## 3. Szybki test – 2 epoki, konfiguracja bazowa

In [ ]:
# Wczytaj dane przez Czetyrba_DataLoader.ipynb LUB z oryginalnego data_loadera
import sys
sys.path.insert(0, '/content/drive/Shareddrives/Sieci_Neuronowe/Projekt_1')
import data_loader as dl

train_loader, test_loader = dl.data_loader(path='./')

# Test podstawowy (batchnorm=False, dropout=False, pool=2, filters=64)
wyniki = run_cnn(
    epochs=2,
    use_batchnorm=False,
    use_dropout=False,
    pool_size=2,
    n_filters=64,
    train_loader=train_loader,
    test_loader=test_loader
)
print(wyniki)

## TODO – do zrobienia przed lekcją
1. Uruchom i sprawdź czy model działa bez błędów
2. Zweryfikuj `flat_size` – upewnij się że wymiary się zgadzają (możliwy błąd dla pool_size≠2)
3. Przetestuj z `use_batchnorm=True` i `use_dropout=True`
4. Porównaj liczbę parametrów dla różnych wartości `n_filters` (32, 64, 128)